### Code for Olivieri PPI analysis
Are the gene hits from refined library provide a better PPI network connectivity than the gene hits from the original library?

In [1]:
import os
import numpy as np
import pandas as pd

In [ ]:
def run_pair_triple(protein_id_1, protein_id_2, pair_name, screen_prefix):
    # === Load HGNC
    hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')
    # === Load STRING
    string = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.links.detailed.v12.0.txt',sep=' ')
    string_info = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.info.v12.0.txt',sep='\t')
    string['protein1'] = string['protein1'].str.replace('9606.', '')
    string['protein2'] = string['protein2'].str.replace('9606.', '')
    string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
    string_info = string_info.iloc[:, :2]
    # === Partner interactions
    p1_edges = string[(string['protein1'] == protein_id_1) | (string['protein2'] == protein_id_1)]
    p2_edges = string[(string['protein1'] == protein_id_2) | (string['protein2'] == protein_id_2)]
    ppi_partners = pd.concat([p1_edges, p2_edges])
    ppi_partners = ppi_partners[ppi_partners['combined_score'] >= 400].sort_values(by='combined_score', ascending=False)
    ppi_partners = ppi_partners.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
    ppi_partners = ppi_partners.rename(columns={'preferred_name': 'gene_symbol_1'}).drop('#string_protein_id', axis=1)
    ppi_partners = ppi_partners.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
    ppi_partners = ppi_partners.rename(columns={'preferred_name': 'gene_symbol_2'}).drop('#string_protein_id', axis=1)
    ppi_partners = ppi_partners[['gene_symbol_1','gene_symbol_2']]
    ppi_partners['SortedInteractors'] = ppi_partners.apply(lambda row: '-'.join(np.sort([row['gene_symbol_1'], row['gene_symbol_2']])), axis=1)
    ppi_partners = ppi_partners.drop_duplicates(subset='SortedInteractors').drop(columns='SortedInteractors').reset_index(drop=True)
    # === HGNC lookup
    gene_lookup = {}
    def normalize_symbol(s): return s.strip().upper() if isinstance(s, str) else None
    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'
        syms = []
        s = normalize_symbol(entry.get('symbol'))
        if s: syms.append(s)
        if pd.notnull(entry.get('prev_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['prev_symbol'].split('|'))
        if pd.notnull(entry.get('alias_symbol')):
            syms.extend(normalize_symbol(x) for x in entry['alias_symbol'].split('|'))
        for sym in syms:
            if sym and sym not in gene_lookup:
                gene_lookup[sym] = (hgnc_id, ensembl_id, entrez_id)
    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))
    ppi_partners['hgnc_id_1'], ppi_partners['ensembl_gene_id_1'], ppi_partners['entrez_id_1'] = zip(*ppi_partners['gene_symbol_1'].apply(lookup_gene_info))
    ppi_partners['hgnc_id_2'], ppi_partners['ensembl_gene_id_2'], ppi_partners['entrez_id_2'] = zip(*ppi_partners['gene_symbol_2'].apply(lookup_gene_info))
    ppi_partners.to_csv(f'results/2_interaction_overlap/string_counts/{screen_prefix}_{pair_name}_string_ppi_partners.csv', index=False)
    # === Load and annotate screen
    screen = pd.read_csv(f'input_data/2_output_with_hgnc/{screen_prefix}_{pair_name}_screen_with_hgnc.csv')
    # Ensure ensembl IDs are strings and drop NaN/empty
    valid_ids = screen['ensembl_gene_id'].fillna("").astype(str).str.strip()
    # Boolean flags for interactions, but only for non-empty IDs
    screen['Interaction_1'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_1'])
    screen['Interaction_2'] = (valid_ids != "") & valid_ids.isin(ppi_partners['ensembl_gene_id_2'])
    # Partner if either column is True
    screen['Interaction_Partners'] = (screen[['Interaction_1', 'Interaction_2']].any(axis=1)).astype(int)
    # === Your slicing logic
    specific_row_3_cell = screen[screen['Cell_line'] == '2 cell line'].index[0] - 1
    specific_row_2_cell = screen[screen['Cell_line'] == '1 cell line'].index[0] - 1
    specific_row_1_cell = screen[screen['Cell_line'] == '0 cell line'].index[0] - 1
    interacted_3_cell_line = screen.loc[:specific_row_3_cell, 'Interaction_Partners'].sum()
    total_3_cell_line = specific_row_3_cell + 1
    interacted_2_cell_line = screen.loc[:specific_row_2_cell, 'Interaction_Partners'].sum() - interacted_3_cell_line
    total_2_cell_line = specific_row_2_cell + 1 - total_3_cell_line
    interacted_1_cell_line = screen.loc[:specific_row_1_cell, 'Interaction_Partners'].sum() - interacted_2_cell_line - interacted_3_cell_line
    total_1_cell_line = specific_row_1_cell + 1 - total_2_cell_line - total_3_cell_line
    interacted_0_cell_line = screen.loc[specific_row_1_cell + 2:, 'Interaction_Partners'].sum()
    total_0_cell_line = len(screen) - (specific_row_1_cell + 1)
    counts_df = pd.DataFrame({
        ' ': [0,1,2,3],
        'Interacted': [interacted_0_cell_line, interacted_1_cell_line, interacted_2_cell_line, interacted_3_cell_line],
        'Total': [total_0_cell_line,total_1_cell_line,total_2_cell_line,total_3_cell_line]
    })
    screen['Interaction_Partners'] = screen['Interaction_Partners'].replace({0: 'Not partner', 1: 'Partner'})
    screen.drop(columns=['Interaction_1','Interaction_2'], inplace=False).to_csv(f'results/2_interaction_overlap/string_results/{screen_prefix}_{pair_name}_string_results.csv', index=False)
    counts_df.to_csv(f"results/2_interaction_overlap/string_counts/{screen_prefix}_{pair_name}_string_count.csv",index=False)
    return ppi_partners, screen, counts_df

### Change the Screen to become our gene hits

In [ ]:
# === Load and annotate screen
ppi_partners, screen_out, counts = run_pair_triple(
    protein_id_1="ENSP00000501150",   # PI3KB
    protein_id_2="ENSP00000361021",   # PTEN
    pair_name="PTEN_PIK3CB",          # used in filenames
    screen_prefix="dunn"              # which screen to load
)